In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [5]:
from sklearn.metrics import classification_report
ths = [19, 20, 21, 22, 23, 24, 25, 26]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 19 hidden = 0


,0,1,2,3,4,5,-1
0,0,189772,0,0,0,3275,9181
1,0,125774,56,1,219,1268,1218
2,0,2,82181,0,0,0,122
3,0,0,3,60923,0,0,24
4,0,76,0,0,45656,14,44
5,0,8,0,0,9,102,28
-1,0,0,0,0,0,0,0


th: 19 hidden: 0 acc:0.6259625814491996 recall:0.04539925232905433 precision:0.8647452199303005 f1:0.08626935093612723
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.86      0.05      0.09    202228
           1       0.40      0.98      0.57    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      1.00      1.00     45790
           5       0.02      0.69      0.04       147

    accuracy                           0.62    519956
   macro avg       0.71      0.79      0.61    519956
weighted avg       0.80      0.62      0.54    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,187281,0,0,0,0,14947
1,0,125201,49,1,207,1264,1814
2,0,2,82121,0,0,0,182
3,0,0,3,60905,0,0,42
4,0,68,0,0,45547,14,161
5,0,5,0,0,9,102,31
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.6355249290324566 recall:0.07391162450303618 precision:0.8701752343249695 f1:0.13625031334746246
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.87      0.07      0.14    202228
           1       0.40      0.97      0.57    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790
           5       0.07      0.69      0.13       147

    accuracy                           0.63    519956
   macro avg       0.72      0.79      0.64    519956
weighted avg       0.80      0.63      0.56    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 24 hidden: 5 acc:0.9411200178476641 recall:0.9115646258503401 precision:0.004359708485163977 f1:0.008677913415147492
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.91      0.01       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.83      0.90    128536
           2       1.00      0.96      0.98     82305
           3       1.00      0.99      1.00     60950
           4       1.00      0.96      0.98     45790

    accuracy                           0.94    519956
   macro avg       0.83      0.94      0.81    519956
weighted avg       1.00      0.94      0.97    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,198382,186,0,0,0,0,3660
1,484,101777,3,0,31,0,26241
2,0,0,75509,0,0,0,6796
3,0,0,3,59268,0,0,1679
4,9,33,0,0,41757,0,3991
5,1,4,0,0,4,0,138
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.9185007962212187 recall:0.9387755102040817 precision:0.0032466768615457007 f1:0.006470974397449124
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.94      0.01       147
           0       1.00      0.98      0.99    202228
           1       1.00      0.79      0.88    128536
           2       1.00      0.92      0.96     82305
           3       1.00      0.97      0.99     60950
           4       1.00      0.91      0.95     45790

    accuracy                           0.92    519956
   macro avg       0.83      0.92      0.80    519956
weighted avg       1.00      0.92      0.95    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,197172,3,0,0,0,0,5053
1,432,70575,3,0,6,0,57520
2,0,0,74050,0,0,0,8255
3,0,0,3,58685,0,0,2262
4,9,31,0,0,39762,0,5988
5,0,4,0,0,3,0,140
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.847900591588519 recall:0.9523809523809523 precision:0.0017672751142417127 f1:0.0035280035280035277
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.95      0.00       147
           0       1.00      0.97      0.99    202228
           1       1.00      0.55      0.71    128536
           2       1.00      0.90      0.95     82305
           3       1.00      0.96      0.98     60950
           4       1.00      0.87      0.93     45790

    accuracy                           0.85    519956
   macro avg       0.83      0.87      0.76    519956
weighted avg       1.00      0.85      0.91    519956



# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 19: 0.3983820250871389
Média f1 absolute th 20: 0.5279468570174457
Média f1 absolute th 21: 0.5589086590141732
Média f1 absolute th 22: 0.5493829931815631
Média f1 absolute th 23: 0.5657439068459161
Média f1 absolute th 24: 0.5658975656379679
Média f1 absolute th 25: 0.5215127259849838
Média f1 absolute th 26: 0.4749664834714812
